# Exploring Uncertainty in Climate Data

There are several kinds of scientific uncertainty that arise when working with long-term projections of future climates:
1. Model Uncertainty, which illustrates the differences between different models (namely, how model physics, settings, or parameters can change the outcome)
2. Scenario Uncertainty, which illustrates the differences in outcomes between end-of-century emissions targets
3. Internal Variability, which represents the variations inherent within the climate system itself

This notebook explores **Model Uncertainty** in the Cal-Adapt: Analytics Engine by focusing on **temperature trends** across simulations. We also compare the suite of models currently available in the [Cal-Adapt Data Catalog](https://analytics.cal-adapt.org/data/) to the full set of CMIP6 models to illustrate the differences between our models and all available models

---
**Story**

As a user, I want to understand when <span style="color:#FF0000">**taking a mean across models is not appropriate for the data**, <span style="color:#000000"> by visualizing the differences between models and see:
1. A range of possibilities for the end of century across all available CMIP6 models
2. Which models are provided in the Analytics Engine as compared to CMIP6 models
3. What the local response (postage stamps) is across models

**Step 0.1** -- Setup and CMIP6 processing (as needed)

**Step 0.2** -- Calculate key metrics needed: (1) Historical baseline (1850-1900); (2) Historical MMM; (3) SSP3-7.0 MMM

**Step 1** -- Illustrate range of temperature trends across CMIP6 archive, in *warming levels* context, with warming level designating line, all historical models + historical MMM, all future models + future MMM +  highlight the Analytics Engine models (one color per model?) amongst spread

**Step 2** -- Illustrate postage stamp view (spatial plots) of min/max/mean/median of CMIP6 for local response looks like
    
**Step 3** -- Illustrate example applications

---

## Step 0: Setup and CMIP6 data processing
Import useful packages and libraries. 

In [ ]:
# !pip install -r requirements.txt
import climakitae as ck
!pip install xmip
import xarray as xr
import pandas as pd
import numpy as np
import datetime
import intake
import dask
import matplotlib as mpl
import matplotlib.pyplot as plt
import xesmf as xe
import cartopy.crs as ccrs
import holoviews as hv
import hvplot.xarray # noqa
import hvplot.pandas

Grab the regridded CMIP6 models that have both historical and SSP3-7.0 simulations

In [ ]:
shp_cat = intake.open_catalog("https://cadcat.s3.amazonaws.com/parquet/catalog.yaml")
us_states = shp_cat.states.read()
us_counties = shp_cat.counties.read()
us_watersheds = shp_cat.huc8.read()
shp_cat = None

col = intake.open_esm_datastore("https://cadcat.s3.amazonaws.com/tmp/cmip6-regrid.json")

In [ ]:
from xmip.preprocessing import (
    rename_cmip6
) # will figure out how to avoid later

def cf_to_dt(ds):
    """
    convert cftime to pandas datetime -
    some GCMs have some wacky calendars
    """
    ds = ds.copy()
    if (
        type(ds.indexes['time']) not in 
        [pd.core.indexes.datetimes.DatetimeIndex]
    ): 
        datetimeindex = ds.indexes['time'].to_datetimeindex()
        ds['time'] = datetimeindex
    return ds

def calendar_align(ds):
    '''
    different models have different calendars
    (some no leap, some 360 day). this results
    in a huge dataset with lots of empty
    values in time when concatenated. 
    the following function sets the day for all monthly
    values to the 1st of each month -
    WARNING this can impact functions which use
    the number of days in each month (eg
    precip flux to total monthly accumulation).
    I can't find a nice way around this.
    '''
    ds['time'] = pd.to_datetime(ds.time.dt.strftime('%Y-%m'))
    return ds

def wrapper(ds):
    ds_simulation = ds.attrs["source_id"]
    ds_scenario = ds.attrs["experiment_id"]
    ds_freq = ds.attrs["frequency"]    
    ds = ds.copy()

    ds = rename_cmip6(ds) # will figure out alternative
    ds = cf_to_dt(ds)
    if ds_freq in ('mon'): 
        ds = calendar_align(ds)
    ds = ds.drop_vars(["lon","lat","height"],
                     errors="ignore")
    ds = ds.assign_coords({'simulation' : 
                           ds_simulation,
                          'scenario' :
                          ds_scenario})
    ds = ds.squeeze(drop=True)
    
    return ds

In [ ]:
class cmip_opt():
    def __init__(self, variable='tas', 
                  area_subset='states', 
                 location='California',
                 timescale='monthly',
                area_average=True):
        self.variable = variable
        self.area_subset = area_subset
        self.location = location
        self.area_average = area_average
        self.timescale = timescale
        
    def cmip_clip(self, ds):
        ds = ds.copy()
        variable = self.variable
        location = self.location
        area_average = self.area_average
        area_subset = self.area_subset
        timescale = self.timescale
        
        to_drop = [v for v in list(
                    ds.data_vars) 
                  if v != variable]
        ds = ds.drop_vars(to_drop)
        ds = clip_region(ds,area_subset,location)
        if area_average:
            ds = area_wgt_average(ds)
        return ds
    
def clip_region(ds,area_subset,location):
    """
    clips CMIP6 dataset using a polygon.
    ds is the dataset
    state is a string ("California")
    (check catalog for other names)
    opt = 'True' to burn in all cells
    which touch the boundary (keep as False)
    """
    ds = ds.copy()
    ds = ds.rio.write_crs(4326)
    
    if 'counties' in area_subset:
        ds_region = us_counties[us_counties.NAME 
                    == location].geometry
    elif 'states' in area_subset:
        ds_region = us_states[us_states.NAME 
                    == location].geometry
        
    try:
        ds = ds.rio.clip(
            geometries=ds_region,crs=4326, drop=True,
        all_touched=False)
    except: 
        #if no grid centers in region
        # instead select all cells which 
        # intersect the region
        print('selecting all cells which intersect region')
        ds = ds.rio.clip(
            geometries=ds_region,crs=4326, drop=True,
        all_touched=True)
        
    return ds

def area_wgt_average(ds):
    ds = ds.copy()
    weights = np.cos(np.deg2rad(ds.y))
    weights.name = "weights"
    ds_weighted = ds.weighted(weights)
    ds = ds_weighted.mean(("x","y"))
    return ds

In [ ]:
# Data selection options
copt = cmip_opt()
copt.variable = 'tas'
copt.area_subset = 'states' 
copt.location = 'California'
copt.area_average = False
copt.timescale = 'monthly'     

In [ ]:
cat = col.search(
    table_id = "Amon",
    variable_id = copt.variable,
    experiment_id = ["historical","ssp370"],
    member_id = "r1i1p1f1",
    require_all_on="source_id"
).search(
    activity_id = ["CMIP","ScenarioMIP"]
)

dsets = cat.to_dataset_dict(
    zarr_kwargs={'consolidated': True}, 
    storage_options={'anon': True},
    preprocess=wrapper
)

# Subsets the historical scenario
hist_dsets = {key: val for key,val in dsets.items()
             if "historical" in key}

# Subsets the future scenario
ssp_dsets = {key: val for key,val in dsets.items()
               if "ssp370" in key}

hist_ds = xr.concat(list(hist_dsets.values()), dim='simulation').squeeze()
hist_ds = copt.cmip_clip(hist_ds.sel(time=slice('1850','2014')))   # mind the historical time slicing

ssp_ds = xr.concat(list(ssp_dsets.values()), dim='simulation').squeeze()
ssp_ds = copt.cmip_clip(ssp_ds)

## Step 1: Assess CMIP6 multi-model spread

In [ ]:
def cmip_annual(ds):
    """Processes CMIP6 dataset into annual smoothed timeseries"""
    ds_degC = ds - 273.15 # convert to degC
    ds_degC = ds_degC.groupby("time.year").mean(dim=["x","y"])
    ds_degC = ds_degC.groupby("time.year").mean(dim="time")
    ds_degC.attrs["units"] = "°C"
    return ds_degC

hist_ds_yr = cmip_annual(hist_ds).compute()
ssp_ds_yr = cmip_annual(ssp_ds).compute()

In [ ]:
# Warming Levels Context [see explore_warming.ipynb]
def calc_anom(ds):
    """Calculates the temperature change relative to a historical baseline, relative to 1850-1900, for each model.
    First calculates the baseline for each input model, using 1850-1900.
    Returns the difference from the input ds and the respective model baseline.
    """
    mdl_baseline = hist_ds_yr.sel(year=slice(1850,1900)).mean("year")
    mdl_temp_anom = ds - mdl_baseline
    return mdl_temp_anom
    
def cmip_mmm(ds):
    """Calculate CMIP6 multi-model mean.
    (1) needed to calculate historical baseline for warming levels
    (2) highlight multi-model mean and model spread
    """
    ds_mmm = ds.mean("simulation")
    return ds_mmm

# Historical
hist_anom = calc_anom(hist_ds_yr.sel(year=slice(1950,2014))) # all historical models
hist_mmm_anom = cmip_mmm(hist_anom) # historical multi-model mean

# Future
ssp_anom = calc_anom(ssp_ds_yr) # all future models
ssp_mmm_anom = cmip_mmm(ssp_anom) # future multi-model mean

Next, plot the temperature spread amongst the members of the CMIP6 archive. 

We also highlight the Cal-Adapt: Analytics Engine models in order to illustrate where these models fall within the larger CMIP6 model spread.

In [ ]:
# Figure set-up
h_color = 'grey'
ssp_color = 'orange'
cae_color = 'blue'
lw = 0.75
alpha = 0.25

# warming level line
warm_level = 3.0
warmlevel_line = hv.HLine(warm_level).opts(color="black", line_width=1.0) \
                * hv.Text(x=1968, y=warm_level+0.45, text=str(warm_level) + "°C warming level").opts(style=dict(text_font_size='8pt'))

# all individual models
all_hist = hist_anom.hvplot.line(x="year", by="simulation", line_width=lw, color=h_color, legend=False, alpha=alpha)
all_ssp = ssp_anom.hvplot.line(x="year", by="simulation", line_width=lw, color=ssp_color, legend=False, alpha=alpha)

# Cal-Adapt: Analytics Engine models
cae_mdls = ["FGOALS-g3", "EC-Earth3-Veg"]
cae_hist_mdls = hist_anom.sel(simulation=cae_mdls)
cae_hist = cae_hist_mdls.hvplot.line(x="year", by="simulation", line_width=lw, color=cae_color, alpha=alpha*1.5)
cae_ssp_mdls = ssp_anom.sel(simulation=cae_mdls)
cae_ssp = cae_ssp_mdls.hvplot.line(x="year", by="simulation", line_width=lw, color=cae_color, alpha=alpha*1.5)

# multi-model means
mmm_hist = hist_mmm_anom.hvplot.line(x="year", line_width=lw*2.5, color='black')
mmm_ssp = ssp_mmm_anom.hvplot.line(x="year", line_width=2.5*lw, color='red',
                                           title="CMIP6 California mean surface temperature change relative to 1850-1900")

all_hist * all_ssp * mmm_hist * mmm_ssp * cae_hist * cae_ssp * warmlevel_line

<span style="color:#FF0000">**Several big notes here**
1. warming level trend is higher than what is seen in the warming levels notebook context figure -- because we are regionally subsetting here
2. only two of the 4 analytics engine models are displayed in the figure above becasue they do not have r1i1p1f1 member_ids (Brian updating)

## Step 2: Illustrate spatial statistics

Next, let's spatially visualize the differences between the CMIP6 model archive at a designated warming level. We will also identiy how the Cal-Aadapt: Analytics Engine models compares to the broader spread. 

First, identify where each individual model reaches the designated warming level. We will look specificallly at a 3°C warming level. 

In [ ]:
# calculate spatial data first
ssp_xy_year = ssp_ds.groupby("time.year").mean(dim="time")
ssp_xy_year = ssp_xy_year - 273.15
ssp_xy_anom = ssp_xy_year - hist_ds_yr.sel(year=slice(1850,1900)).mean("year")

In [ ]:
# find the year each warming level is reached
warmlevel = 3.0

# determine where each model reaches the warming level
da_list = []
year_reached_by_sim = []
for simulation in ssp_anom.simulation.values: 
    data_to_use = ssp_anom.tas.sel(simulation=simulation)
    years_above_warmlevel = data_to_use.where(data_to_use > warmlevel, drop=True)
    if len(years_above_warmlevel.year.values) < 1: 
        print("Warming level not reached for {0}".format(simulation))
    else: 
        year_warmlevel_reached = years_above_warmlevel.year.values[0]
        year_reached_by_sim.append((simulation, year_warmlevel_reached))
        da_list.append(ssp_xy_anom.sel(year=year_warmlevel_reached, simulation=simulation))
    
thresh_df = pd.DataFrame(
    data=year_reached_by_sim, 
    columns=["simulation","year_warming_level_reached"]
)
data_by_warmlevel = xr.concat(da_list, dim="simulation")

In [ ]:
# Set up for plots
from climakitae.utils import _read_ae_colormap
cmap = _read_ae_colormap(cmap='ae_diverging', cmap_hex=True) # sets the colormap to ae_diverging

def _make_hvplot(data, title, width=200, height=200):
    """Make single map"""
    _plot = data.hvplot.image(
        x="x",
        y="y",
        grid=True,
        width=width,
        height=height,
        xaxis=None,
        yaxis=None,
        title="{}".format(data.simulation.values),
        # clabel="{0} ({1})".format(data.tas.long_name, data.tas.units),
        features=["coastline"],
        cmap=cmap,
    )
    return _plot

Now, let's plot the difference (anomaly) between the year that the warming level is reached and the 1850-1900 historical baseline for each simulation. Please note, this will take a few minutes to plot!

In [ ]:
num_simulations = len(data_by_warmlevel.simulation.values) # number of simulations

all_plots = _make_hvplot(  # Need to make the first plot separate from the loop
        data=data_by_warmlevel.isel(simulation=0),
        title=data_by_warmlevel.isel(simulation=0).simulation.item())

for sim_i in range(1, num_simulations):
    pl_i = _make_hvplot(
        data=data_by_warmlevel.isel(simulation=sim_i),
        title=data_by_warmlevel.isel(simulation=sim_i).simulation.item())
    all_plots += pl_i

all_plots.cols(8)  # Organize into 6 columns
all_plots.opts(hv.opts.Layout(merge_tools=True))  # Merge toolbar

all_plots

<span style="color:#FF0000">**To do**
1. Standardize data plot range (vmin, vmax)
2. Single color bar with descriptive label (air temperature °C)
3. Ensure colormap is centered around 0

Next, let's see visualize the minimum/maximum/median/mean conditions across models. These statistics are calculated from the data observed in the figure above at each grid cell. 

In [ ]:
# Compute stats
min_data = data_by_warmlevel.min(dim="simulation")
max_data = data_by_warmlevel.max(dim="simulation")
med_data = data_by_warmlevel.median(dim="simulation")
mean_data = data_by_warmlevel.mean(dim="simulation")

def _make_stats_hvplot(data, title, width=300, height=300):
    """Make single map"""
    _plot = data.hvplot.image(
        x="x",
        y="y",
        grid=True,
        width=width,
        height=height,
        features=["coastline"],
        cmap=cmap)
    return _plot

# Set up plots
min_plot = _make_stats_hvplot(
    data=min_data,
    title="Minimum", width=width, height=height)

max_plot = _make_stats_hvplot(
    data=max_data,
    title="Maximum", width=width, height=height)

med_plot = _make_stats_hvplot(
    data=med_data,
    title="Median", width=width, height=height)

mean_plot = _make_stats_hvplot(
    data=mean_data,
    title="Mean", width=width, height=height)

In [ ]:
all_plots = mean_plot + med_plot + min_plot + max_plot
all_plots.opts(
    title="Air Temperature Anomalies for "
    + str(warmlevel)
    + "°C Warming Across Models"
)
all_plots.opts(toolbar="below")  # Set toolbar location
all_plots.opts(hv.opts.Layout(merge_tools=True))  # Merge toolbar
    
all_plots

<span style="color:#FF0000">**To do**
1. identify why individual plot titles are not included
2. center color bar at 0 (or single colorbar)

## Step 3: Example applications

What might be most useful to know when addressing model uncertainty in climate data, is to know when it is **not appropriate to use an average across models**, and when it is **reasonable to do so** (note this depends on the question being asked)

First, we will assess when it is not appropriate to use a multi-model mean in climate data. 

In [ ]:
# outline of idea when taking a mean is bad:  "I want to know what the exact temperature will be at my location in 2100"

# Step 1. Specific location in california - perhaps central california where there is a lot of variation in temp anomaly (location that has neg temp change in some models and pos in others)
# Step 2. Provide a line plot of all model possibillities and the multi-model mean, highlight that there is a several degree difference (+ standard deviation around mean)
# Step 3. Provide histogram(?) of range to illustrate the tails + standard deviations
# Step 4/Info. Discuss that there are some models with "too much warming" and some that are "too little warming"

# caveat: this approaches localizing to a point, which we want to potentially avoid

Now let's look at when it is scientifically reasonable to take an average across models in climate data. 

In [ ]:
# outline of idea when taking a mean is okay: "Will Southern California be warmer in the future than it was in the past?"

# Step 1. look at Southern Cal. 
# Step 2. Use baseline historical period (can do 1850-1900 bc of warming levels) and future period of reference (maybe 2070-2100?)
# Step 3. Calculate difference between these two periods to illustrate that agreement across models is future > historical, and multi-model mean is okay

# need to have a further think if this is the most illustrative/useful example of okay to use multi-model mean

<span style="color:#FF0000">**To do**
1. Needs detailed markdown describing/explaining uncertainty context for each figure. To be done once major code dev is completed. 
2. Standardize mapping functions (_make_hvplot and _make_stats_hvplot) into a single function so it does not have to be defined twice